# 02 — SDXL-Lightning Image Processor
Only generates images that do not already exist.

In [ ]:
import os,sys,subprocess
ROOT="/content/main"
REPO_URL="https://github.com/jonbBla/black-history-factory.git"  # put your GitHub repo URL here if you want automatic git clone
if not os.path.exists(ROOT):
    if not REPO_URL: raise RuntimeError("Set REPO_URL to your GitHub repository before running.")
    subprocess.run(["git","clone",REPO_URL,ROOT],check=True)
sys.path.insert(0,ROOT)
from factory.drive import mount_drive,DrivePaths
from factory.config import Config
MYDRIVE=mount_drive(); paths=DrivePaths(os.path.join(MYDRIVE,"BLACK_HISTORY_FACTORY")); paths.ensure_tree(); config=Config.load(paths.root)
print(paths.root)


In [ ]:
from factory.image_engine import load_sdxl_lightning,run
from factory.utils import read_json,write_json_atomic
from factory import status
pipe=load_sdxl_lightning()
while True:
    jobs=[j for j in os.listdir(paths("02_JOBS")) if (read_json(paths.manifest(j),{}) or {}).get("status") in ("QWEN_READY","IMAGES_PARTIAL")]
    if not jobs: print("No image jobs available."); break
    for job_id in jobs:
        try:
            scenes=read_json(paths.scenes(job_id),[]); status.set_processor(paths,"image","running",job_id,f"{len(scenes)} scenes")
            run(paths,job_id,scenes,pipe,config)
            d=read_json(paths.manifest(job_id),{}); d["status"]="IMAGES_READY"; write_json_atomic(paths.manifest(job_id),d)
            status.set_processor(paths,"image","idle",job_id,"ready"); print("IMAGES READY",job_id)
        except Exception as e:
            d=read_json(paths.manifest(job_id),{}); d.update(status="IMAGES_PARTIAL",image_error=str(e)); write_json_atomic(paths.manifest(job_id),d); status.set_processor(paths,"image","error",job_id,str(e))
